In [1]:
# loading raw data
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
wts= pd.read_csv(PROCESSED_DIR/ "daily_weight_reconstruction.csv")
gt_29=wts[wts['weight']>29.7]
gt_29=gt_29.groupby(['symbol','weight_source']).size().reset_index(name="count")
gt_29
anomaly=wts[wts['weight']>29.7]
anomaly['date']=pd.to_datetime(anomaly['date'])
months_anom=anomaly['date'].dt.strftime("%Y-%m").unique()
months_anom


C:\Users\musaib\AppData\Local\Temp\ipykernel_20000\580909375.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anomaly['date']=pd.to_datetime(anomaly['date'])


array(['2023-07', '2023-09', '2024-04', '2025-08', '2026-01'],
      dtype=object)

In [4]:
off_wts=pd.read_csv(RAW_DIR/ "constituent_weights_raw.csv")
off_wts
months_anom=pd.PeriodIndex(months_anom,freq="M")
months_to_extract = months_anom.union(months_anom - 1)
filtrd_wts= off_wts[off_wts['source_month'].isin(months_to_extract.astype(str))]

pd.set_option('display.max_rows',None)
filtrd_wts[['date','symbol','close_price','weightage_pct','source_month','weight_observation_type']]

,date,symbol,close_price,weightage_pct,source_month,weight_observation_type
60,2023-06-30,AUBANK,753.65,2.61,2023-06,MONTH_END
61,2023-06-30,AXISBANK,987.45,10.17,2023-06,MONTH_END
62,2023-06-30,BANDHANBNK,242.05,1.30,2023-06,MONTH_END
63,2023-06-30,BANKBARODA,190.35,2.56,2023-06,MONTH_END
64,2023-06-30,FEDERALBNK,126.15,1.93,2023-06,MONTH_END
65,2023-06-30,HDFCBANK,1701.40,27.74,2023-06,MONTH_END
66,2023-06-30,ICICIBANK,934.60,24.13,2023-06,MONTH_END
67,2023-06-30,IDFCFIRSTB,79.40,1.79,2023-06,MONTH_END
68,2023-06-30,INDUSINDBK,1374.65,6.47,2023-06,MONTH_END
69,2023-06-30,KOTAKBANK,1846.55,10.02,2023-06,MONTH_END


In [5]:

WEIGHT_FILE = RAW_DIR  / "constituent_weights_raw.csv"
STOCK_FILE = RAW_DIR / "constituent_stocks_raw.csv"


# ---------------------------------------------------------
# LOAD RAW DATA
# ---------------------------------------------------------
official = pd.read_csv(RAW_DIR/ "constituent_weights_raw.csv")
stocks= pd.read_csv(RAW_DIR/ "constituent_stocks_raw.csv")

official["date"] = pd.to_datetime(official["date"])
stocks["date"] = pd.to_datetime(stocks["date"])


# ---------------------------------------------------------
# RECONSTRUCTION FUNCTION
# ---------------------------------------------------------

def reconstruct_month(
    anchor_date,
    start_date,
    end_date,
):
    """
    Independently reconstruct daily constituent weights
    starting from an official anchor.

    Formula:

        w_i,t =
            w_i,t-1 * (P_i,t / P_i,t-1)
            --------------------------------
            sum_j [w_j,t-1 * (P_j,t / P_j,t-1)]

    The official anchor weights are used only at anchor_date.
    """

    anchor_date = pd.Timestamp(anchor_date)
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    # ---------------------------------------------
    # Official anchor
    # ---------------------------------------------

    anchor = official[
        (official["date"] == anchor_date)
        & (official["weight_observation_type"].isin(
            ["MONTH_END", "REBALANCE_EFFECTIVE"]
        ))
    ].copy()

    anchor = anchor[
        ["symbol", "weightage_pct"]
    ].copy()

    anchor["weight"] = anchor["weightage_pct"] / 100

    anchor = anchor[["symbol", "weight"]]

    symbols = anchor["symbol"].tolist()

    print(f"\nAnchor: {anchor_date.date()}")
    print(f"Constituents: {len(symbols)}")
    print(f"Anchor weight sum: {anchor['weight'].sum():.10f}")

    # ---------------------------------------------
    # Stock prices
    # Include anchor date because we need
    # P_t / P_(t-1)
    # ---------------------------------------------

    price = stocks[
        (stocks["symbol"].isin(symbols))
        & (stocks["date"] >= anchor_date)
        & (stocks["date"] <= end_date)
    ].copy()

    price = (
        price
        .pivot(
            index="date",
            columns="symbol",
            values="close"
        )
        .sort_index()
    )

    # Keep only dates in requested range
    price = price.loc[
        (price.index >= anchor_date)
        & (price.index <= end_date)
    ]

    # ---------------------------------------------
    # Start with official anchor
    # ---------------------------------------------

    weights = anchor.set_index("symbol")["weight"].copy()

    results = []

    # Save anchor
    for symbol in symbols:
        results.append({
            "date": anchor_date,
            "symbol": symbol,
            "weight": weights[symbol],
            "weight_source": "OFFICIAL"
        })

    # ---------------------------------------------
    # Forward reconstruction
    # ---------------------------------------------

    previous_date = anchor_date

    for current_date in price.index:

        if current_date <= anchor_date:
            continue

        previous_prices = price.loc[previous_date]
        current_prices = price.loc[current_date]

        returns = current_prices / previous_prices

        # Weighted market-cap proxy
        numerator = weights * returns

        denominator = numerator.sum()

        new_weights = numerator / denominator

        weights = new_weights

        for symbol in symbols:
            results.append({
                "date": current_date,
                "symbol": symbol,
                "weight": weights[symbol],
                "weight_source": "INDEPENDENT_RECONSTRUCTION"
            })

        previous_date = current_date

    result = pd.DataFrame(results)

    return result


# ---------------------------------------------------------
# AUGUST 2025
# ---------------------------------------------------------

aug_2025 = reconstruct_month(
    anchor_date="2025-07-31",
    start_date="2025-08-01",
    end_date="2025-08-29",
)


# ---------------------------------------------------------
# JANUARY 2026
# ---------------------------------------------------------

jan_2026 = reconstruct_month(
    anchor_date="2025-12-31",
    start_date="2026-01-01",
    end_date="2026-01-30",
)


# ---------------------------------------------------------
# COMBINE
# ---------------------------------------------------------

independent_reconstruction = pd.concat(
    [aug_2025, jan_2026],
    ignore_index=True
)

independent_reconstruction = (
    independent_reconstruction
    .sort_values(["date", "symbol"])
    .reset_index(drop=True)
)

print(
    "\nIndependent reconstruction complete."
)

print(
    independent_reconstruction[
        independent_reconstruction["symbol"].isin(
            ["HDFCBANK", "KOTAKBANK"]
        )
    ].to_string(index=False)
)


Anchor: 2025-07-31
Constituents: 12
Anchor weight sum: 1.0001000000

Anchor: 2025-12-31
Constituents: 14
Anchor weight sum: 1.0000000000

Independent reconstruction complete.
      date    symbol   weight              weight_source
2025-07-31  HDFCBANK 0.290900                   OFFICIAL
2025-07-31 KOTAKBANK 0.078300                   OFFICIAL
2025-08-01  HDFCBANK 0.291802 INDEPENDENT_RECONSTRUCTION
2025-08-01 KOTAKBANK 0.079326 INDEPENDENT_RECONSTRUCTION
2025-08-04  HDFCBANK 0.288863 INDEPENDENT_RECONSTRUCTION
2025-08-04 KOTAKBANK 0.079507 INDEPENDENT_RECONSTRUCTION
2025-08-05  HDFCBANK 0.288117 INDEPENDENT_RECONSTRUCTION
2025-08-05 KOTAKBANK 0.080119 INDEPENDENT_RECONSTRUCTION
2025-08-06  HDFCBANK 0.288973 INDEPENDENT_RECONSTRUCTION
2025-08-06 KOTAKBANK 0.080021 INDEPENDENT_RECONSTRUCTION
2025-08-07  HDFCBANK 0.289867 INDEPENDENT_RECONSTRUCTION
2025-08-07 KOTAKBANK 0.079452 INDEPENDENT_RECONSTRUCTION
2025-08-08  HDFCBANK 0.289435 INDEPENDENT_RECONSTRUCTION
2025-08-08 KOTAKBANK 0.078

In [6]:
proc=pd.read_csv("../data/processed/constituent_stocks_data.csv")
proc['Date']=pd.to_datetime(proc['Date'])
bank_nifty=pd.read_csv("../data/raw/bank_nifty_daily_raw.csv")
bank_nifty['date']=pd.to_datetime(bank_nifty['date'],format="%d-%m-%Y")
bank_nifty['date']=bank_nifty['date'].dt.strftime("%Y-%m-%d")
bank_nifty['date']=pd.to_datetime(bank_nifty['date'])

weekday_const=proc[proc['Date'].dt.dayofweek < 5 ]
weekday_bnf=bank_nifty[bank_nifty['date'].dt.dayofweek < 5]
missin_in_proc= set(weekday_bnf['date']) - set(weekday_const['Date'])
missin_in_bnf = set(weekday_const['Date']) - set(weekday_bnf['date'])

missin_in_bnf==missin_in_proc

False

In [7]:
display(missin_in_bnf)
display(missin_in_proc)

{Timestamp('2023-04-06 00:00:00'),
 Timestamp('2023-04-10 00:00:00'),
 Timestamp('2023-04-11 00:00:00'),
 Timestamp('2026-01-15 00:00:00'),
 Timestamp('2026-05-01 00:00:00'),
 Timestamp('2026-05-28 00:00:00'),
 Timestamp('2026-06-26 00:00:00')}

set()

In [9]:
import os
os.getcwd()

'c:\\Users\\musaib\\Desktop\\banknifty_event_impact\\notebooks'

In [11]:
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.dates_validation import get_missing_dates
from src.preprocessing.missing_data_fix import ( fetch_missing_data)

missing_dates = get_missing_dates(
    reference_df=proc,
    target_df=bank_nifty,
    reference_date_col ="Date",
    target_date_col = 'date'
)

new_banknifty_rows= fetch_missing_data(ticker="^NSEBANK",missing_dates=missing_dates)

$^NSEBANK: possibly delisted; no price data found  (1d 2026-01-15 00:00:00 -> 2026-01-16 00:00:00)

1 Failed download:
['^NSEBANK']: possibly delisted; no price data found  (1d 2026-01-15 00:00:00 -> 2026-01-16 00:00:00)
$^NSEBANK: possibly delisted; no price data found  (1d 2026-05-01 00:00:00 -> 2026-05-02 00:00:00)

1 Failed download:
['^NSEBANK']: possibly delisted; no price data found  (1d 2026-05-01 00:00:00 -> 2026-05-02 00:00:00)
$^NSEBANK: possibly delisted; no price data found  (1d 2026-05-28 00:00:00 -> 2026-05-29 00:00:00)

1 Failed download:
['^NSEBANK']: possibly delisted; no price data found  (1d 2026-05-28 00:00:00 -> 2026-05-29 00:00:00)
$^NSEBANK: possibly delisted; no price data found  (1d 2026-06-26 00:00:00 -> 2026-06-27 00:00:00)

1 Failed download:
['^NSEBANK']: possibly delisted; no price data found  (1d 2026-06-26 00:00:00 -> 2026-06-27 00:00:00)


In [12]:
new_banknifty_rows

Price,Date,Adj Close,Close,High,Low,Open,Volume
Ticker,,^NSEBANK,^NSEBANK,^NSEBANK,^NSEBANK,^NSEBANK,^NSEBANK
0,2023-04-06,41041.000000,41041.000000,41274.699219,40820.550781,40940.699219,196500
1,2023-04-10,40834.648438,40834.648438,41139.851562,40727.250000,41116.148438,153900
2,2023-04-11,41366.500000,41366.500000,41403.250000,40990.199219,41232.000000,219200


In [14]:
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from src.cleaning.remove_dates import remove_dates

holiday_dates=[
    "2026-01-15","2026-05-01","2026-05-08","2026-06-26"
]

display(proc.shape)
proc_cleaned = remove_dates(df=proc,dates=holiday_dates,date_column='Date')
display(proc_cleaned.shape)


(13605, 7)

(13545, 7)

In [16]:
proc_cleaned.to_csv(
    "../data/processed/constituent_stocks_data.csv",
    index=False
)

In [24]:
from src.cleaning.remove_dates import remove_dates

ohlc_stocks_equal= proc[(proc['Open'] == proc['High']) & (proc['High']==proc['Low']) & (proc['Low']==proc['Close'])]
display(ohlc_stocks_equal.shape)

display(proc.shape)
proc_cleaned = remove_dates(df=proc,dates=ohlc_stocks_equal['Date'],date_column='Date')
display(proc_cleaned.shape)

display()
proc_cleaned.to_csv(
    "../data/processed/constituent_stocks_data.csv",
    index=False
)


(75, 7)

(13605, 7)

(13530, 7)

In [32]:
from src.cleaning.standardize_dataset import standardize_dataset
df_cleaned=standardize_dataset("../data/processed/constituent_stocks_data.csv","../data/processed/constituent_stocks_data.csv",{'Ticker':"symbol"},['Date'])


In [52]:
bank_nifty=pd.read_csv("../data/processed/bank_nifty_daily_amended.csv")
bank_nifty['date']=pd.to_datetime(bank_nifty['date'],format="%d-%m-%Y")
bank_nifty['date']=bank_nifty['date'].dt.strftime("%Y-%m-%d")
bank_nifty['date']=pd.to_datetime(bank_nifty['date'])

bank_nifty['date'].dtype

dtype('<M8[ns]')

In [53]:
bank_nifty.to_csv("../data/processed/bank_nifty_daily_amended.csv",index=False)

In [66]:
stocks=pd.read_csv("../data/processed/constituent_stocks_data.csv")

stocks['symbol']=stocks['symbol'].str.split(".").str[0]
stocks.to_csv("../data/processed/constituent_stocks_data.csv",index=False)


In [73]:
wt_val =pd.read_csv("../data/processed/weight_reconstruction_validation_constituent.csv")

wt_val.head(5)

,validation_type,month_end,reconstructed_date,official_comparison_date,symbol,reconstructed_weight,official_weight,deviation,absolute_deviation,percentage_deviation,squared_error
0,NORMAL,2023-02-28,2023-02-01,2023-01-31,AUBANK,2.348662,2.49,-0.141338,0.141338,-5.676205,0.019976
1,NORMAL,2023-02-28,2023-02-01,2023-01-31,AXISBANK,9.694740,10.01,-0.315260,0.315260,-3.149454,0.099389
2,NORMAL,2023-02-28,2023-02-01,2023-01-31,BANDHANBNK,1.449815,1.55,-0.100185,0.100185,-6.463525,0.010037
3,NORMAL,2023-02-28,2023-02-01,2023-01-31,BANKBARODA,2.585798,2.62,-0.034202,0.034202,-1.305410,0.001170
4,NORMAL,2023-02-28,2023-02-01,2023-01-31,FEDERALBNK,2.284248,2.38,-0.095752,0.095752,-4.023181,0.009168


In [84]:
comparing=wt_val[['official_comparison_date','symbol','reconstructed_weight','official_weight','percentage_deviation']]
comparing['official_comparison_date']=pd.to_datetime(comparing['official_comparison_date'])
for date, group in comparing.groupby('official_comparison_date'):
    print(f"\n { date.date()}")
    display(group)


 2023-01-31


C:\Users\musaib\AppData\Local\Temp\ipykernel_20000\1145217033.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comparing['official_comparison_date']=pd.to_datetime(comparing['official_comparison_date'])


,official_comparison_date,symbol,reconstructed_weight,official_weight,percentage_deviation
0,2023-01-31,AUBANK,2.348662,2.49,-5.676205
1,2023-01-31,AXISBANK,9.694740,10.01,-3.149454
2,2023-01-31,BANDHANBNK,1.449815,1.55,-6.463525
3,2023-01-31,BANKBARODA,2.585798,2.62,-1.305410
4,2023-01-31,FEDERALBNK,2.284248,2.38,-4.023181
5,2023-01-31,HDFCBANK,28.525970,28.66,-0.467655
6,2023-01-31,ICICIBANK,24.758958,23.54,5.178239
7,2023-01-31,IDFCFIRSTB,1.393967,1.49,-6.445161
8,2023-01-31,INDUSINDBK,6.179399,5.91,4.558352
9,2023-01-31,KOTAKBANK,10.237043,10.18,0.560341



 2023-02-28


,official_comparison_date,symbol,reconstructed_weight,official_weight,percentage_deviation
12,2023-02-28,AUBANK,2.334637,2.41,-3.127086
13,2023-02-28,AXISBANK,10.071880,9.79,2.879260
14,2023-02-28,BANDHANBNK,1.067626,1.48,-27.863138
15,2023-02-28,BANKBARODA,2.860858,2.50,14.434316
16,2023-02-28,FEDERALBNK,2.446955,2.31,5.928808
17,2023-02-28,HDFCBANK,28.095419,28.86,-2.649277
18,2023-02-28,ICICIBANK,24.666265,24.42,1.008457
19,2023-02-28,IDFCFIRSTB,1.505213,1.42,6.000897
20,2023-02-28,INDUSINDBK,5.989816,5.94,0.838661
21,2023-02-28,KOTAKBANK,9.940498,10.27,-3.208392


...

,official_comparison_date,symbol,reconstructed_weight,official_weight,percentage_deviation
489,2026-05-29,AUBANK,4.607661,4.87,-5.386836
490,2026-05-29,AXISBANK,9.386508,10.28,-8.691556
491,2026-05-29,BANKBARODA,3.948808,4.47,-11.659773
492,2026-05-29,CANBK,3.112545,3.98,-21.795340
493,2026-05-29,FEDERALBNK,7.088680,6.38,11.107843
494,2026-05-29,HDFCBANK,19.561296,17.93,9.098139
495,2026-05-29,ICICIBANK,14.576939,13.63,6.947463
496,2026-05-29,IDFCFIRSTB,4.514066,4.27,5.715835
497,2026-05-29,INDUSINDBK,4.756923,5.40,-11.908838
498,2026-05-29,KOTAKBANK,8.979757,9.81,-8.463228



 2026-06-30


,official_comparison_date,symbol,reconstructed_weight,official_weight,percentage_deviation
503,2026-06-30,AUBANK,4.740615,4.64,2.168422
504,2026-06-30,AXISBANK,7.995030,9.59,-16.631596
505,2026-06-30,BANKBARODA,3.227403,4.00,-19.314921
506,2026-06-30,CANBK,3.316906,3.31,0.208633
507,2026-06-30,FEDERALBNK,7.968509,6.67,19.467897
508,2026-06-30,HDFCBANK,17.264510,19.30,-10.546581
509,2026-06-30,ICICIBANK,15.605032,14.16,10.205032
510,2026-06-30,IDFCFIRSTB,5.057602,4.36,16.000057
511,2026-06-30,INDUSINDBK,5.969788,4.99,19.635034
512,2026-06-30,KOTAKBANK,9.185533,9.31,-1.336920



 2026-07-31


,official_comparison_date,symbol,reconstructed_weight,official_weight,percentage_deviation
517,2026-07-31,AUBANK,4.948970,4.71,5.073677
518,2026-07-31,AXISBANK,9.421238,8.81,6.937997
519,2026-07-31,BANKBARODA,3.364940,3.58,-6.007272
520,2026-07-31,CANBK,3.331380,3.32,0.342777
521,2026-07-31,FEDERALBNK,6.898611,7.29,-5.368849
522,2026-07-31,HDFCBANK,16.057381,18.20,-11.772634
523,2026-07-31,ICICIBANK,14.828398,14.86,-0.212662
524,2026-07-31,IDFCFIRSTB,4.701454,4.67,0.673527
525,2026-07-31,INDUSINDBK,5.395570,5.50,-1.898733
526,2026-07-31,KOTAKBANK,10.445088,9.32,12.071763


In [89]:
TOP_banks= ['HDFCBANK','ICICIBNK']
filtrd_bnks=comparing[comparing['symbol'].isin(TOP_banks)]

net_dev= filtrd_bnks.groupby('official_comparison_date', as_index=False)['percentage_deviation'].sum()
display(net_dev)

,official_comparison_date,percentage_deviation
0,2023-01-31,-0.467655
1,2023-02-28,-2.649277
2,2023-03-31,-2.746374
3,2023-04-28,-11.962795
4,2023-05-31,12.013697
5,2023-06-30,-2.829134
6,2023-07-31,-3.568501
7,2023-08-31,-0.174209
8,2023-09-29,2.004529
9,2023-10-31,3.110136


In [95]:
#Chronological ordering of bank_nifty
bank_nifty=bank_nifty.sort_values('date').reset_index(drop=True)
bank_nifty.to_csv("../data/processed/bank_nifty_daily_amended.csv",index=False,date_format="%Y-%m-%d")

In [98]:
bnf=pd.read_csv("../data/processed/bank_nifty_daily_amended.csv")
bnf['date']=pd.to_datetime(bnf['date'])
bnf.to_csv("../data/processed/bank_nifty_daily_amended.csv",index=False)
display(bnf['date'].head(15))
display(bnf.head(5))
display(bnf['date'].dtype)

0    2023-01-02
1    2023-01-03
2    2023-01-04
3    2023-01-05
4    2023-01-06
5    2023-01-09
6    2023-01-10
7    2023-01-11
8    2023-01-12
9    2023-01-13
10   2023-01-16
11   2023-01-17
12   2023-01-18
13   2023-01-19
14   2023-01-20
Name: date, dtype: datetime64[ns]

,date,open,high,low,close
0,2023-01-02,43038.25,43382.75,42961.75,43203.10
1,2023-01-03,43151.45,43482.80,43148.45,43425.25
2,2023-01-04,43417.50,43578.40,42858.65,42958.80
3,2023-01-05,43073.05,43137.55,42298.60,42608.70
4,2023-01-06,42649.70,42685.90,41877.00,42188.80


dtype('<M8[ns]')

In [102]:
from src.cleaning.standardize_dataset import standardize_dataset

df_cleaned=standardize_dataset("../data/processed/constituent_regime_members.csv","../data/processed/constituent_regime_members.csv",column_renames=None,date_columns=['start_snapshot_date'])


In [106]:

df_cleaned=standardize_dataset("../data/processed/constituent_regimes.csv","../data/processed/constituent_regimes.csv",column_renames=None,date_columns=['start_snapshot_date','end_snapshot_date'])


In [109]:

df_cleaned=standardize_dataset("../data/processed/daily_weight_reconstruction_new.csv","../data/processed/daily_weight_reconstruction_new.csv",column_renames=None,date_columns=['date'])


In [111]:
df_cleaned=standardize_dataset("../data/processed/weight_reconstruction_validation_constituent.csv","../data/processed/weight_reconstruction_validation_constituent.csv",column_renames=None,date_columns=['month_end','reconstructed_date','official_comparison_date'])


In [115]:
df_cleaned=standardize_dataset("../data/processed/weight_reconstruction_validation_monthly.csv","../data/processed/weight_reconstruction_validation_monthly.csv",column_renames=None,date_columns=['month_end','reconstructed_date','official_comparison_date'])


In [117]:
df_cleaned=standardize_dataset("../data/raw/constituent_change_events.csv","../data/processed/constituent_change_events.csv",column_renames=None,date_columns=['effective_date'])


In [119]:
df_cleaned=standardize_dataset("../data/raw/constituent_weights_raw.csv","../data/processed/constituent_weights_raw.csv",column_renames=None,date_columns=['date','source_month'])


In [121]:
df_cleaned=standardize_dataset("../data/raw/earnings_raw.csv","../data/processed/earnings.csv",column_renames=None,date_columns=['date','period_end'])


c:\Users\musaib\Desktop\banknifty_event_impact\src\cleaning\standardize_dataset.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[column] = pd.to_datetime(


In [ ]:
from src.cleaning.convert_revenue import convert_revenue
earn=pd.read_csv("..data/processed/earnings.csv")
earn['eps_actual']=earn['eps_actual'].replace('--',np.nan)
earn['revenue_actual']=earn['revenue_actual'].replace('--',np.nan)
earn['eps_surprise']=earn['eps_surprise'].replace('--',np.nan)
earn['revenue_surprise']=earn['revenue_surprise'].replace('--',np.nan)
earn['eps_forecast']=earn['eps_forecast'].replace("--",np.nan)
earn['reaction']=earn['reaction'].replace("--",np.nan)
earn['revenue_forecast']=earn['revenue_forecast'].replace('--',np.nan)

earn['eps_surprise']=earn['eps_surprise'].str.replace('%',"",regex=False).astype(float)
earn['revenue_surprise']=earn['revenue_surprise'].str.replace('%',"",regex=False).astype(float)
earn['reaction']=earn['reaction'].str.replace('%',"",regex=False).astype(float)

earn[['revenue_actual','revenue_forecast']]=(earn[['revenue_actual','revenue_forecast']].apply(lambda col: col.map(convert_revenue)))
earn['eps_actual']=earn['eps_actual'].astype(float)
earn['eps_forecast']=earn['eps_forecast'].astype(float)
earn['date']=pd.to_datetime(earn['date'])
earn['period_end']=pd.to_datetime(earn['period_end'])
earn.to_csv("../data/processed/earnings.csv")
